In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, current_date

#pip install pyspark jupyterlab --break-system-packages

#создаёт Spark-сессию с автоматической загрузкой JAR-зависимостей для работы с внешними системами
spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()

:: loading settings :: url = jar:file:/home/coder/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6b4bc683-a15e-4606-a506-d05ca5a7bb87;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickho

In [4]:
# Параметры подключения к PostgreSQL
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")
jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
properties = {
    "user": db_user,
    "password": db_password,
    "driver": "org.postgresql.Driver"
}

table_name = "public.shops"
shops = spark.read.jdbc(
    jdbc_url,
    table_name,
    properties=properties
).select("st_id", "shop_name")

#shops.show()
shops.show(5)
shops.printSchema()

#

+-----+-----------+
|st_id|  shop_name|
+-----+-----------+
|  842|      Lenta|
|  843|     Magnit|
|  844|       Spar|
|  845|Pyaterochka|
|  846|      Lenta|
+-----+-----------+
only showing top 5 rows
root
 |-- st_id: integer (nullable = true)
 |-- shop_name: string (nullable = true)



In [5]:
from pyspark.sql.functions import current_date
#трансформация
df = shops.select(
        col("st_id"),
        current_date().alias("date"),
        col("shop_name")
        ,lit(1).alias("version")
    )

df.show(10)
print(f"Записей: {df.count()}")

+-----+----------+-----------+-------+
|st_id|      date|  shop_name|version|
+-----+----------+-----------+-------+
|  842|2026-02-02|      Lenta|      1|
|  843|2026-02-02|     Magnit|      1|
|  844|2026-02-02|       Spar|      1|
|  845|2026-02-02|Pyaterochka|      1|
|  846|2026-02-02|      Lenta|      1|
|  847|2026-02-02|      Diksi|      1|
|  848|2026-02-02|      Lenta|      1|
|  849|2026-02-02|   FixPrice|      1|
|  850|2026-02-02|     Magnit|      1|
|  851|2026-02-02|      Lenta|      1|
+-----+----------+-----------+-------+

Записей: 10


In [ ]:
#Запись в клик
jdbc_url = 'jdbc:clickhouse://clickhouse01:8123/shushiato'
jdbc_url_dev = 'jdbc:clickhouse://clickhouse01:8123/dev'
jdbc_driver = "com.clickhouse.jdbc.ClickHouseDriver"
db_user = os.getenv('CLICKHOUSE_USER')
db_password = os.getenv('CLICKHOUSE_PASSWORD')
database = 'shushiato'
table_name = 'shops_dist'

#чтение максимальной версии и запись с версией больше

max_version_df = spark.read \
    .format("jdbc") \
    .option("driver", jdbc_driver) \
    .option("url", jdbc_url) \
    .option("dbtable", f"(SELECT coalesce(max(version), 0) as max_version FROM {table_name}) AS tmp") \
    .option("user", db_user) \
    .option("password", db_password) \
    .load()

max_version = int(max_version_df.collect()[0]['max_version'])
# print(f"Максимальная версия: {max_version} (int)")

# Добавляем в DataFrame
df = df.withColumn("version", lit(max_version + 1))
df.show(10)



Максимальная версия: 1 (int)
+-----+----------+-----------+-------+
|st_id|      date|  shop_name|version|
+-----+----------+-----------+-------+
|  842|2026-02-02|      Lenta|      2|
|  843|2026-02-02|     Magnit|      2|
|  844|2026-02-02|       Spar|      2|
|  845|2026-02-02|Pyaterochka|      2|
|  846|2026-02-02|      Lenta|      2|
|  847|2026-02-02|      Diksi|      2|
|  848|2026-02-02|      Lenta|      2|
|  849|2026-02-02|   FixPrice|      2|
|  850|2026-02-02|     Magnit|      2|
|  851|2026-02-02|      Lenta|      2|
+-----+----------+-----------+-------+

Максимальная версия: 1 (int)


In [8]:


df.write \
    .format("jdbc") \
    .option("driver", jdbc_driver) \
    .option("url", jdbc_url) \
    .option("dbtable", table_name) \
    .option("user", db_user) \
    .option("password", db_password) \
    .mode("append") \
    .save()


print("✅ Данные загружены в ClickHouse!")

✅ Данные загружены в ClickHouse!


26/02/02 10:02:36 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
26/02/02 10:02:36 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction is not supported. Change jdbcCompliant to false to throw SQLException instead.
26/02/02 10:02:36 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction [898ea332-796d-4a42-91d8-5071066d614b](2 queries & 0 savepoints) is committed.
26/02/02 10:02:36 WARN ClickHouseConnectionImpl: [JDBC Compliant Mode] Transaction [65d96c0d-2b2f-4c5a-86c5-79af1705a77b](0 queries & 0 savepoints) is committed.


In [9]:
spark.stop()